# Day 2 · 信贷违约 EDA

数据：UCI Default of Credit Card Clients，30000 x 25。

流程：加载 -> 4 张图 -> 数据质量发现 -> WOE 建模宽表落盘。

**运行方式**：`Restart & Run All`，必须能从头跑完。

In [ ]:
import os
import pathlib

import pandas as pd
from IPython.display import Image, display

# 让相对路径稳定地相对项目根——notebook 的默认工作目录是 notebooks/
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
print("project root:", ROOT)

from credit.data_loader import clean, load_raw, split_xy
from credit.features import build_iv_table

df = clean(load_raw())
X, y = split_xy(df)
print("shape:", df.shape)
print("missing total:", int(df.isna().sum().sum()))
print("default rate:", round(float(y.mean()), 4))

## 1. 四张图

每张图回答一个问题：

1. 目标分布 —— 类别平衡吗？
2. 缺失情况 —— 有多少缺失？（总数 0 也是一条结论）
3. 额度五分位 vs 违约率 —— 单调吗？
4. 学历档 vs 违约率 —— 有没有异常档？

In [ ]:
from credit import eda

eda.run_all()  # 出 4 张图 + 落盘 data/processed/features.csv 与 reports/iv_table.csv

for name in [
    "01_target_distribution",
    "02_missing",
    "03_limit_quintile_vs_default",
    "04_education_vs_default",
]:
    display(Image(filename=f"reports/figures/{name}.png"))

## 2. 数据质量发现：EDUCATION 的未声明类别值

UCI 文档只声明了 `EDUCATION ∈ {1,2,3,4}`，但数据里出现了 `0 / 5 / 6`。

下面这段在【不做任何处理】和【把 0/5/6 并入 4(others)】两种情况下，
分别算 EDUCATION 的箱内坏率与 IV，用来量化「处理前后」的差异。

In [ ]:
raw_stat = (
    pd.DataFrame({"education": X["education"], "y": y})
    .groupby("education")["y"]
    .agg(n="size", bad_rate="mean")
)
raw_stat["share"] = (raw_stat["n"] / raw_stat["n"].sum()).round(5)
raw_stat["bad_rate"] = raw_stat["bad_rate"].round(4)
print("=== 处理前：原始 EDUCATION 档次 ===")
print(raw_stat.to_string())
print()
print("=== 处理前 IV ===")
print(build_iv_table(X[["education"]], y).to_string(index=False))

In [ ]:
X_fixed = X.copy()
X_fixed["education"] = X_fixed["education"].replace({0: 4, 5: 4, 6: 4})

fixed_stat = (
    pd.DataFrame({"education": X_fixed["education"], "y": y})
    .groupby("education")["y"]
    .agg(n="size", bad_rate="mean")
)
fixed_stat["share"] = (fixed_stat["n"] / fixed_stat["n"].sum()).round(5)
fixed_stat["bad_rate"] = fixed_stat["bad_rate"].round(4)
print("=== 处理后：0/5/6 并入 others(4) ===")
print(fixed_stat.to_string())
print()
print("=== 处理后 IV ===")
print(build_iv_table(X_fixed[["education"]], y).to_string(index=False))

## 3. 数据质量发现（记录，四要素）

> 下面前三项**你来写**，第四项用上面两个 cell 的实际输出填。

- **现象**：⬜ 
- **原因**：⬜ 
- **怎么处理**：⬜ 
- **处理前后的数字**：⬜ 

### 4 条可执行结论

> 每条要写成「该做什么」，不是「数据显示…」。给 Day3 建模用。

1. ⬜ 
2. ⬜ 
3. ⬜ 
4. ⬜ 